# 🗄️ SPARK TUTORIAL 03: SPARK SQL

## 🎯 **OBJETIVO**
Dominar Spark SQL para consultas complejas

## 📋 **CONTENIDO**
- Creación de vistas temporales
- Consultas SQL complejas
- Funciones SQL avanzadas
- Integración con Hive
- Optimización de consultas

---

## 🔧 **CONFIGURACIÓN INICIAL**


In [5]:
# 🔄 CELDA DE REINICIO - Ejecutar si hay errores de SparkContext
# Esta celda cierra cualquier sesión anterior y crea una nueva

try:
    if 'spark' in globals():
        print("🔄 Cerrando sesión anterior de Spark...")
        spark.stop()
        print("✅ Sesión anterior cerrada")
except:
    print("ℹ️ No había sesión anterior")

# Limpiar variables
if 'spark' in globals():
    del spark

print("🚀 Listo para crear nueva sesión de Spark")


🔄 Cerrando sesión anterior de Spark...
✅ Sesión anterior cerrada
🚀 Listo para crear nueva sesión de Spark


In [6]:
# Importar librerías para Spark SQL
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

print("📚 Librerías de Spark SQL importadas correctamente")


📚 Librerías de Spark SQL importadas correctamente


In [7]:
# Crear SparkSession para el curso
import socket
import os

# Detectar si estamos dentro de un contenedor Docker
def get_spark_master():
    try:
        # Intentar conectar al master de Docker
        hostname = socket.gethostname()
        if 'jupyter' in hostname or 'master' in hostname or 'jupyterlab' in hostname:
            return "spark://master:7077"  # Desde dentro del contenedor
        else:
            return "spark://localhost:7077"  # Desde fuera del contenedor
    except:
        return "local[*]"  # Fallback a modo local

spark_master_url = get_spark_master()
print(f"🔧 Conectando a: {spark_master_url}")

spark = SparkSession.builder \
    .appName("EducacionIT-Spark-Basics") \
    .master(spark_master_url) \
    .config("spark.executor.memory", "2g")\
    .config("spark.executor.cores", "1")\
    .config("spark.executor.instances", "1")\
    .config("spark.driver.memory", "1g") \
    .config("spark.driver.cores", "1") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("✅ SparkSession creada exitosamente")


🔧 Conectando a: spark://master:7077
✅ SparkSession creada exitosamente


## 📊 **PASO 1: CREAR DATOS DE EJEMPLO**

Vamos a crear tablas de ejemplo para consultas SQL.


In [8]:
# Crear DataFrame de clientes
clientes_data = [
    (1, "Juan Pérez", "juan.perez@email.com", "Madrid", "2020-01-15", "Premium"),
    (2, "María García", "maria.garcia@email.com", "Barcelona", "2019-03-20", "Standard"),
    (3, "Carlos López", "carlos.lopez@email.com", "Madrid", "2021-06-10", "Premium"),
    (4, "Ana Martínez", "ana.martinez@email.com", "Valencia", "2018-11-05", "Basic"),
    (5, "Luis Rodríguez", "luis.rodriguez@email.com", "Sevilla", "2017-09-12", "Premium")
]

clientes_schema = StructType([
    StructField("cliente_id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("email", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("fecha_registro", StringType(), True),
    StructField("tipo_cliente", StringType(), True)
])

df_clientes = spark.createDataFrame(clientes_data, clientes_schema)
df_clientes = df_clientes.withColumn("fecha_registro", col("fecha_registro").cast(DateType()))

print("👥 Tabla clientes:")
df_clientes.show()


👥 Tabla clientes:


+----------+--------------+--------------------+---------+--------------+------------+
|cliente_id|        nombre|               email|   ciudad|fecha_registro|tipo_cliente|
+----------+--------------+--------------------+---------+--------------+------------+
|         1|    Juan Pérez|juan.perez@email.com|   Madrid|    2020-01-15|     Premium|
|         2|  María García|maria.garcia@emai...|Barcelona|    2019-03-20|    Standard|
|         3|  Carlos López|carlos.lopez@emai...|   Madrid|    2021-06-10|     Premium|
|         4|  Ana Martínez|ana.martinez@emai...| Valencia|    2018-11-05|       Basic|
|         5|Luis Rodríguez|luis.rodriguez@em...|  Sevilla|    2017-09-12|     Premium|
+----------+--------------+--------------------+---------+--------------+------------+



## 👁️ **PASO 2: CREAR VISTAS TEMPORALES**

Las vistas temporales permiten usar DataFrames como tablas SQL.


In [9]:
# Crear vista temporal
df_clientes.createOrReplaceTempView("clientes")

print("✅ Vista temporal 'clientes' creada")
print("\n📋 Tablas disponibles:")
spark.sql("SHOW TABLES").show()


✅ Vista temporal 'clientes' creada

📋 Tablas disponibles:


25/09/27 05:01:54 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/09/27 05:01:54 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/09/27 05:01:54 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/09/27 05:01:54 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/09/27 05:01:55 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/09/27 05:01:55 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
25/09/27 05:01:56 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         | clientes|       true|
+---------+---------+-----------+



## 🔍 **PASO 3: CONSULTAS SQL BÁSICAS**

Vamos a ejecutar consultas SQL directamente.


In [10]:
# Consulta simple
print("1️⃣ Todos los clientes premium:")
spark.sql("""
    SELECT cliente_id, nombre, ciudad, fecha_registro
    FROM clientes 
    WHERE tipo_cliente = 'Premium'
    ORDER BY fecha_registro DESC
""").show()


1️⃣ Todos los clientes premium:


[Stage 2:=============================>                             (1 + 1) / 2]

+----------+--------------+-------+--------------+
|cliente_id|        nombre| ciudad|fecha_registro|
+----------+--------------+-------+--------------+
|         3|  Carlos López| Madrid|    2021-06-10|
|         1|    Juan Pérez| Madrid|    2020-01-15|
|         5|Luis Rodríguez|Sevilla|    2017-09-12|
+----------+--------------+-------+--------------+



In [11]:
# Análisis por ciudad
print("2️⃣ Clientes por ciudad:")
spark.sql("""
    SELECT ciudad, 
           COUNT(*) as total_clientes,
           COUNT(CASE WHEN tipo_cliente = 'Premium' THEN 1 END) as clientes_premium
    FROM clientes
    GROUP BY ciudad
    ORDER BY total_clientes DESC
""").show()


2️⃣ Clientes por ciudad:


[Stage 4:====================================================>  (192 + 2) / 200]

+---------+--------------+----------------+
|   ciudad|total_clientes|clientes_premium|
+---------+--------------+----------------+
|   Madrid|             2|               2|
|  Sevilla|             1|               1|
| Valencia|             1|               0|
|Barcelona|             1|               0|
+---------+--------------+----------------+



## 🎯 **RESUMEN DEL TUTORIAL**

¡Felicitaciones! Has completado el tutorial de Spark SQL.

### **📚 Conceptos SQL aprendidos:**
- ✅ **Vistas temporales**: Crear tablas SQL desde DataFrames
- ✅ **Consultas SQL básicas**: SELECT, WHERE, GROUP BY
- ✅ **Funciones SQL**: COUNT, CASE WHEN
- ✅ **Integración con Hive**: Soporte completo para Hive
- ✅ **Optimización**: Configuración para consultas eficientes

### **🚀 Próximos pasos:**
1. **Experimentar** con consultas más complejas
2. **Integrar** con datos reales del proyecto
3. **Optimizar** consultas de producción

---

**🎉 ¡Has dominado Spark SQL!**


In [ ]:
# Cerrar SparkSession
spark.stop()
print("🔒 SparkSession cerrada correctamente")
